In [ ]:
# 1. Load a CSV and print how many missing values are in each column.
# 2. Remove exact duplicate rows from a DataFrame.
# 3. Given a `hotel_name_raw` column with inconsistent casing/spacing, write a
#    function that maps it to a canonical `hotel_id` using a lookup dict.
# 4. Calculate total revenue per room type.
# 5. Find the top 3 guests by total spend.
# 6. Add a column categorizing `booking_channel` into 'Online' (Direct, OTA)
#    vs 'Offline' (Corporate) — use a dict lookup with `.map()`.
# 7. Merge a `bookings` DataFrame with a `guests` DataFrame and count how many
#    rows failed to match (i.e., a guest_id with no match).
# 8. Write a function `is_valid_email(email: str) -> bool` using basic string
#    checks (contains "@", contains "." after the "@", no spaces).

In [1]:
import pandas as pd

In [15]:
# 1. Load a CSV and print how many missing values are in each column.  
hotels_df = pd.read_csv('data/hotels_raw.csv')
print(hotels_df.isnull().sum())

bookings_df = pd.read_csv('data/bookings_raw.csv')
print(bookings_df.isnull().sum())

guests_df = pd.read_csv('data/guests_raw.csv')
print(guests_df.isnull().sum())

rooms_df = pd.read_csv('data/rooms_raw.csv')
print(rooms_df.isnull().sum())



hotel_id      0
hotel_name    0
city          0
country       0
dtype: int64
booking_id              0
guest_id                0
hotel_name_raw          0
room_type_id            0
check_in                0
check_out               0
nights                  0
rate_per_night          0
total_revenue           0
satisfaction_score    185
booking_channel         0
dtype: int64
guest_id          0
first_name        0
last_name         0
email             0
country           0
loyalty_member    0
dtype: int64
room_type_id      0
room_type_name    0
base_rate         0
dtype: int64


In [16]:
# 2. Remove exact duplicate rows from a DataFrame.
hotels_df = hotels_df.drop_duplicates() 
guests_df = guests_df.drop_duplicates()
rooms_df = rooms_df.drop_duplicates()
bookings_df = bookings_df.drop_duplicates()

In [19]:
# 3. Given a `hotel_name_raw` column with inconsistent casing/spacing, write a
#    function that maps it to a canonical `hotel_id` using a lookup dict.

def normalize_name(name: str) -> str:
    return ' '.join(name.strip().lower().split())


def build_hotel_lookup(hotel_names, hotel_ids):
    return {normalize_name(name): hid for name, hid in zip(hotel_names, hotel_ids)}

def map_to_hotel_id(hotel_name_raw: str, lookup: dict):
    return lookup.get(normalize_name(hotel_name_raw))

hotel_lookup = build_hotel_lookup(hotels_df['hotel_name'], hotels_df['hotel_id'])
bookings_df['hotel_id'] = bookings_df['hotel_name_raw'].apply(lambda x: map_to_hotel_id(x, hotel_lookup))

In [20]:
# 4. Calculate total revenue per room type.
total_revenue_per_room_type = bookings_df.groupby('room_type_id')['total_revenue'].sum().reset_index()
print(total_revenue_per_room_type)


   room_type_id  total_revenue
0             1      354149.45
1             2      463872.57
2             3      610392.54
3             4      298188.78


In [21]:
# 5. Find the top 3 guests by total spend.
top_3_guests = bookings_df.groupby('guest_id')['total_revenue'].sum().nlargest(3).reset_index()
print(top_3_guests)

   guest_id  total_revenue
0       208       13324.43
1       114       10608.61
2       248        9732.22


In [23]:
# 6. Add a column categorizing `booking_channel` into 'Online' (Direct, OTA)
#    vs 'Offline' (Corporate) — use a dict lookup with `.map()`.
create_dict = {
    'Direct': 'Online',
    'OTA': 'Online',
    'Corporate': 'Offline'
}
bookings_df['booking_channel_category'] = bookings_df['booking_channel'].map(create_dict)

bookings_df.head()


,booking_id,guest_id,hotel_name_raw,room_type_id,check_in,check_out,nights,rate_per_night,total_revenue,satisfaction_score,booking_channel,hotel_id,booking_channel_category
0,1,258,Boston North Station,2,2025-06-07,2025-06-09,2,165.17,330.34,5.0,Direct,5,Online
1,2,393,Boston North Station,4,2025-02-01,2025-02-05,4,106.23,424.92,5.0,Direct,5,Online
2,3,31,London Bankside,2,2026-03-27,2026-03-31,4,136.44,545.76,4.0,Corporate,2,Offline
3,4,67,rotterdam,3,2026-06-12,2026-06-17,5,188.38,941.90,5.0,Direct,6,Online
4,5,128,copenhagen islands brygge,2,2025-12-02,2025-12-07,5,135.65,678.25,3.0,Corporate,7,Offline


In [24]:
# 7. Merge a `bookings` DataFrame with a `guests` DataFrame and count how many
#    rows failed to match (i.e., a guest_id with no match).
merged_df = bookings_df.merge(guests_df, on='guest_id', how='left', indicator=True)
failed_matches_count = merged_df[merged_df['_merge'] == 'left_only'].shape[0]
print(f"Number of rows that failed to match: {failed_matches_count}")

Number of rows that failed to match: 0


In [25]:
# 8. Write a function `is_valid_email(email: str) -> bool` using basic string
#    checks (contains "@", contains "." after the "@", no spaces).
def is_valid_email(email: str) -> bool:
    if "@" not in email or " " in email:
        return False
    local_part, domain_part = email.split("@", 1)
    if "." not in domain_part:
        return False
    return True